## Milvus

In [170]:
from langchain_milvus import Milvus, BM25BuiltInFunction
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import TokenTextSplitter, CharacterTextSplitter, RecursiveCharacterTextSplitter
from pymilvus import Collection, MilvusException, connections, db, utility, MilvusClient

from uuid import uuid4
import numpy as np
import pymupdf
import tiktoken
import dropbox
import re

#### Embeddings

In [2]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### Vector Database

In [3]:
conn = connections.connect(host="localhost", port=19530)

# Create a new database
db_name = "test"
try:
    database = db.create_database(db_name)
    print(f"Database '{db_name}' created successfully.")
except:
    print(f"Database '{db_name}' already exists or could not be created.")

2025-10-12 03:24:57,756 [ERROR][handler]: RPC error: [create_database], <MilvusException: (code=65535, message=database already exist: test)>, <Time:{'RPC start': '2025-10-12 03:24:57.713218', 'RPC error': '2025-10-12 03:24:57.756912'}> (decorators.py:140)


Database 'test' already exists or could not be created.


In [4]:
client = MilvusClient(
    db_name='document_embeddings'
)

In [5]:
client.use_database('document_embeddings')

In [6]:
client.list_collections()

['LangChainCollection']

#### Vector search

In [211]:
URI = 'http://localhost:19530'

vector_store = Milvus(
    embedding_function=embeddings,
    connection_args={"uri": URI, 'token': 'root:Milvus', 'db_name': 'document_embeddings'},
    index_params={"index_type": "FLAT", "metric_type": "L2"},
    consistency_level="Strong",
    drop_old=False
)

In [212]:
retriever = vector_store.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 3, 'fetch_k': 100, 'lambda_mult': 0.5}
)

In [213]:
results = retriever.invoke(
    'values of kmod',
    k=30
)

len(results)

30

In [215]:
for res in results:
    print(res.metadata['page_start'], res.metadata['page_end'], res.metadata['page_labels'], res.metadata['pk'])

57 58  9c4c877b-3d0f-4360-b2c3-07d012965e8a
26 27  f4770a06-78d7-431d-ac6c-761d1d431cb8
68 70 38;39 14e0ec5a-2d3f-41f5-b634-498eeef100a9
150 153 120;121;122 4dac0b21-dabe-4450-a906-2a719246643e
46 47 15;16 31dec932-fad3-43cd-aa3c-68e5826ced46
50 51  8391b5fb-1c9b-4879-82ce-01c8cdaae985
12 14  b5c22121-df2c-4e54-8ecf-87e7eeea6a56
87 89 57;58 aa3d0fc1-e7e2-4ed2-babf-9a7985c13511
75 76 44;45 593e3bbd-1f7a-424c-8109-3981cebfa915
161 163 5;6 e415dfa5-521c-4e23-8043-d60c1ac4a092
131 132 100;101 9951a196-271d-459f-9130-dbe66ff76122
80 81 49;50 83fbe8c5-3c13-483a-ac3b-f762c74e05cb
105 106  d19f521f-2d0e-472e-928c-fa41c4487a5d
54 55 23;24 78d3e486-264a-4b02-ab30-8098067862cd
83 84 52;53 9f6f9a90-45d1-4528-85a9-109432a5b2da
92 93  6b3111b6-a972-4434-88c0-33bc6393fe23
51 52 20;21 66bc4a4c-e785-421a-a3b1-5f226949a617
119 120 88;89 c76e8387-b59c-4be2-834b-baf40db8e7e2
143 147 114;115;116 ceafece8-643c-401e-a7fd-d746a6d5c0df
22 24 10;11 93b10a4f-e959-424b-a7a5-c45d6035f0cf
19 20  638f41de-eb50-40f7-

#### Langchain document loaders

In [11]:
pdf_loader = PyPDFLoader('ns-en-1995-1-1_2004+a2_2014+na_2024_en_001.pdf')

In [12]:
s = pdf_loader.load_and_split()

In [13]:
s[60]

Document(metadata={'producer': 'https://RealtaOnline.com - https://AntennaHouse.com - 20240309-1540z ; modified using iText® 7.1.12 ©2000-2020 iText Group NV (AGPL-version)', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2024-09-03T16:12:51+00:00', 'source': 'ns-en-1995-1-1_2004+a2_2014+na_2024_en_001.pdf', 'total_pages': 168, 'page': 60, 'page_label': '61'}, page_content='EN 1995-1-1:2004 (E) \n 29\nh\ns\nhk\n⎧⎛⎞⎪⎜⎟⎪⎝⎠= ⎨\n⎪\n⎪⎩\n300\nmin\n1, 2\n (3.3) \nwhere: \nh  is the depth of the member, in mm; \ns is the size effect exponent, refer to 3.4(5)P. \n \n(4) The reference length in tension is 3000 mm. For lengths in tension not equal to 3000 mm the \ncharacteristic value for \nfB\nt,0,k\nB should be multiplied by the factor kB\nA \nBgiven by \ns\nk\n⎧⎛⎞⎪⎜⎟⎪⎝⎠= ⎨\n⎪\n⎪⎩\nA\n/2\n3000\nmin\n1, 1\nA  (3.4) \nwhere A is the length, in mm. \n \n(5)P The size effect exponent s for LVL shall be taken as declared in accordance with \nEN 14374. \n \n(6)P Large finger joints complying wit

In [25]:
doc = pymupdf.open('ns-en-1995-1-1_2004+a2_2014+na_2024_en_001.pdf')

new_doc = pymupdf.open()

In [ ]:
rec: pymupdf.Rect = pymupdf.Rect(42, 72, 563, 772)

In [36]:
page = doc[0]
width, height = page.mediabox_size

total_height = height * len(doc)

In [37]:
new_page = new_doc.new_page(width=width, height=total_height)

In [38]:
y_offset = 0
for page in doc:
    page.clip_to_rect(rec)
    pix = page.get_pixmap(dpi=200)
    img_rect = pymupdf.Rect(0, y_offset, width, y_offset + height)
    new_page.insert_image(img_rect, pixmap=pix)
    y_offset += height

In [39]:
new_doc.save('output.pdf')

In [44]:
loader = PyPDFLoader('output.pdf')

In [45]:
chunks = loader.load_and_split()

In [46]:
chunks

[]

#### Page merge with PyPDF

In [77]:
from pypdf import PdfReader, PdfWriter, PageObject

doc = PdfReader(open('ns-en-1995-1-1_2004+a2_2014+na_2024_en_001.pdf', 'rb'))
new_doc = PdfWriter()

In [62]:
page = doc.pages[0]
width, height = page.mediabox.width, page.mediabox.height
total_height = height * len(doc.pages)

width, height, total_height

(595.276, 841.89, 141437.52)

In [66]:
new_page = PageObject.create_blank_page(None, width, total_height)

In [67]:
y_offset = 0
for page in doc.pages:
    new_page.merge_translated_page(page, 0, y_offset)
    y_offset += height

In [78]:
new_doc.add_page(new_page)
new_doc.write('output2.pdf')

(True, <_io.FileIO [closed]>)

In [79]:
loader = PyPDFLoader('output2.pdf')

In [80]:
chunks = loader.load_and_split()

In [81]:
chunks[20]

Document(metadata={'producer': 'pypdf', 'creator': 'PyPDF', 'creationdate': '', 'source': 'output2.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='EN´s for construction products relevant to timber structures \nEN 1998 “Design of structures for earthquake resistance”, when timber structures are built in \nseismic regions \n \n(4) EN 1995 is subdivided into various parts: \nEN 1995-1 General \nEN 1995-2 Bridges \n \n(5) EN 1995-1 “General” comprises: \n EN 1995-1-1 General  – Common rules and rules for buildings \n EN 1995-1-2 General rules – Structural Fire Design \n \n(6) EN 1995-2 refers to the common rules in EN 1995-1-1. The clauses in EN 1995-2 \nsupplement the clauses in EN 1995-1. \n \n1.1.2 Scope of EN 1995-1-1 \n \n(1) EN 1995-1-1 gives general design rules for timber structures together with specific design \nrules for buildings. \n \n(2) The following subjects are dealt with in EN 1995-1-1: \nSection 1: General \nSection 2: Basis of design \nSection 3: Ma

In [216]:
def extract_page_label(page: pymupdf.Page) -> str|None:
    if page.number % 2:
        label = page.get_textbox(pymupdf.Rect(42, 784, 100, 808)).strip()
    else:
        label = page.get_textbox(pymupdf.Rect(400, 784, 564, 808)).strip()

    match = re.search(r'^\d{1,3}', label)
    
    return match

In [ ]:
doc = pymupdf.open('ns-en-1995-1-1_2004+a2_2014+na_2024_en_001.pdf')

rec: pymupdf.Rect = pymupdf.Rect(42, 72, 563, 772)
tokenizer = tiktoken.get_encoding('cl100k_base')

chunk_size = 800
chunk_overlap = 400

token_buffer = []
page_buffer = []
page_labels = []
for page in doc:
    page_label = extract_page_label(page)
    if not page_label:
        continue

    page_label = page_label.group(0)

    text = page.get_textbox(rec).strip()
    tokens = tokenizer.encode(text)
    token_buffer.extend(tokens)
    page_buffer.append(page.number)
    page_labels.append(page_label)

    while len(token_buffer) >= chunk_size + chunk_overlap:
        chunk_tokens = token_buffer[:chunk_size + chunk_overlap]
        chunk_content = tokenizer.decode(chunk_tokens)

        # Create document from chunk
        metadata = {
            'Document': 'ns-en-1995-1-1_2004+a2_2014+na_2024_en_001.pdf',
            'pages': ';'.join(map(str, page_buffer)),
            'page_labels': ';'.join(page_labels),
        }
        document = Document(
            page_content=chunk_content,
            metadata=metadata
        )

        # Remove used tokens from buffer, keeping overlap tokens
        del token_buffer[:chunk_size + chunk_overlap // 2]

        print(len(chunk_tokens), len(token_buffer), page_buffer, page_labels)

        if len(token_buffer):
            page_buffer = page_buffer[-1:]
            page_labels = page_labels[-1:]
        else:
            page_buffer = []
            page_labels = []

print(len(token_buffer), page_buffer, page_labels)

{'Document': 'ns-en-1995-1-1_2004+a2_2014+na_2024_en_001.pdf', 'pages': '5;6;7', 'page_labels': '2;3;4'}
1200 230 [5, 6, 7] ['2', '3', '4']
{'Document': 'ns-en-1995-1-1_2004+a2_2014+na_2024_en_001.pdf', 'pages': '7;8;9;10', 'page_labels': '4;5;6;7'}
1200 659 [7, 8, 9, 10] ['4', '5', '6', '7']
{'Document': 'ns-en-1995-1-1_2004+a2_2014+na_2024_en_001.pdf', 'pages': '10;11;12', 'page_labels': '7;8;9'}
1200 683 [10, 11, 12] ['7', '8', '9']
{'Document': 'ns-en-1995-1-1_2004+a2_2014+na_2024_en_001.pdf', 'pages': '12;13;15', 'page_labels': '9;10;2'}
1200 216 [12, 13, 15] ['9', '10', '2']
{'Document': 'ns-en-1995-1-1_2004+a2_2014+na_2024_en_001.pdf', 'pages': '15;16;17', 'page_labels': '2;3;4'}
1200 422 [15, 16, 17] ['2', '3', '4']
{'Document': 'ns-en-1995-1-1_2004+a2_2014+na_2024_en_001.pdf', 'pages': '17;18;19', 'page_labels': '4;5;6'}
1200 879 [17, 18, 19] ['4', '5', '6']
{'Document': 'ns-en-1995-1-1_2004+a2_2014+na_2024_en_001.pdf', 'pages': '19;20', 'page_labels': '6;7'}
1200 513 [19, 20]